# 0. Imports

In [2]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import mannwhitneyu

# 1. Constants

In [3]:
DATA_DIR = Path("data/")
COHORTS  = ["cohort_1", "cohort_2"]

LEARNING_ITEMS = {
    "ml":                       "L1. Machine learning",
    "ml_types":                 "L2. Three ML paradigms and their differences",
    "ml_usage":                 "L3. Where each ML paradigm can be applied",
    "ml_data_importance":       "L4. Data and its importance in ML",
    "nearest_neighbor":         "L5. KNN",
    "linear_regression":        "L6. Linear regression",
    "reinforcement_learning":   "L7. Reinforcement learning",
    "exploration_exploitation": "L8. Exploration-exploitation dilemma",
}

COURSE_DESIGN_ITEMS = {
    "course_design_math":                     "D1. I was able to follow the mathematics",
    "course_design_clear_visualization":      "D2. The data visualizations were clear",
    "course_design_interesting_visualization":"D3. The data visualizations were interesting",
    "course_design_web_platform":             "D4. The web platform is intuitive and easy to use",
    "course_design_presentation":             "D5. The presentations and examples were easy to understand",
    "course_engagement_recommend":            "D6. I would recommend this course to my friends",
    "course_engagement_fun":                  "D7. I found the devices we built fun and interesting",
}

MOTIVATION_ITEMS = {
    "course_impact_continue": "E1. I want to continue exploring ML after the course",
    "course_impact_skills":   "E2. The skills learned will be useful in future studies or career",
}

# 2. Code

In [4]:
# Data loading

def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def records_to_df(records, cohort) -> pd.DataFrame:
    rows = [{"cohort": cohort, "submission_id": r["submission_id"], **r["responses"]}
            for r in records]
    return pd.DataFrame(rows)


def load_all():
    pre_frames, post_frames = [], []
    for cohort in COHORTS:
        pre  = load_json(DATA_DIR / cohort / "pre.json")
        post = load_json(DATA_DIR / cohort / "post.json")
        pre_frames.append(records_to_df(pre, cohort))
        post_frames.append(records_to_df(post, cohort))
    return pd.concat(pre_frames, ignore_index=True), pd.concat(post_frames, ignore_index=True)


# Helpers 

def col(df, key):
    return df[key].dropna()


def describe(s):
    return len(s), s.mean(), s.std()


def mwu(pre, post):
    _, p = mannwhitneyu(pre.values, post.values, alternative="two-sided")
    return p


def get_significance_stars(p: float) -> str:
    if p <= 0.01: return "**"
    if p <= 0.05: return "*"
    return ""


# Printers

def print_learning_outcomes(pre_df, post_df):
    print("\n" + "="*80)
    print("LEARNING OUTCOMES  (self-assessed Likert 1-5, Mann-Whitney U, two-sided)")
    print("="*80)
    print(f"{'Concept':<52} {'n_pre':>5} {'M_pre(SD)':>11} {'n_post':>6} {'M_post(SD)':>11} {'p':>7}")
    print("-"*80)
    for key, label in LEARNING_ITEMS.items():
        pre_s  = col(pre_df,  key)
        post_s = col(post_df, key)
        n_pre,  m_pre,  sd_pre  = describe(pre_s)
        n_post, m_post, sd_post = describe(post_s)
        p = mwu(pre_s, post_s)
        stars = get_significance_stars(p)
        p_str = f"{p:.3f}"
        print(f"{label+stars:<52} {n_pre:>5} {m_pre:>5.2f}({sd_pre:.2f}) "
              f"{n_post:>6} {m_post:>5.2f}({sd_post:.2f}) {p_str:>7}")
    print("\n* p <= 0.05   ** p <= 0.01")


def print_post_ratings(post_df):
    print("\n" + "="*80)
    print("COURSE DESIGN & MOTIVATION  (post-survey, Likert 1-5)")
    print("="*80)
    print(f"{'Question':<62} {'n':>4} {'M(SD)':>9}")
    print("-"*80)
    for key, label in {**COURSE_DESIGN_ITEMS, **MOTIVATION_ITEMS}.items():
        s = col(post_df, key)
        n, m, sd = describe(s)
        print(f"{label:<62} {n:>4} {m:>5.2f}({sd:.2f})")


def print_career_interest(pre_df, post_df):
    print("\n" + "="*80)
    print("CAREER INTEREST IN AI/ML/ROBOTICS")
    print("="*80)
    for label, df in [("Pre-survey", pre_df), ("Post-survey", post_df)]:
        counts = df["career_interest"].dropna().explode().value_counts()
        n = counts.sum()
        print(f"{label} (n={n}):  "
              f"yes={counts.get('yes', 0)}  "
              f"no={counts.get('no', 0)}  "
              f"undecided={counts.get('undecided', 0)}")


def print_interest_areas(pre_df, post_df):
    print("\n" + "="*80)
    print("INTEREST AREAS")
    print("="*80)
    for label, df in [("Pre-survey", pre_df), ("Post-survey", post_df)]:
        if "interest_areas" not in df.columns:
            continue
        counts = df["interest_areas"].explode().value_counts()
        print(f"\n{label} (n={len(df)}):")
        for area, cnt in counts.items():
            print(f"  {area}: {cnt} ({cnt/len(df)*100:.0f}%)")


def print_word_associations(pre_df, post_df):
    print("\n" + "="*80)
    print("WORD ASSOCIATIONS WITH 'AI'")
    print("="*80)
    for label, df in [("Pre-survey", pre_df), ("Post-survey", post_df)]:
        words = df["word_associations"].dropna().explode().dropna().str.strip()
        words = words[words != ""]
        print(f"\n{label} ({len(words)} words total):")
        print("  " + ", ".join(sorted(words.tolist())))


def print_open_feedback(post_df):
    print("\n" + "="*80)
    print("OPEN FEEDBACK")
    print("="*80)
    for _, row in post_df.iterrows():
        fb = row.get("open_feedback")
        if fb and str(fb).strip():
            print(f"  [{row['cohort']} / {row['submission_id']}] {fb}")


# Main

def main():
    pre_df, post_df = load_all()
    print(f"Loaded {len(pre_df)} pre-survey and {len(post_df)} post-survey responses.")

    print_learning_outcomes(pre_df, post_df)
    print_post_ratings(post_df)
    print_career_interest(pre_df, post_df)
    print_interest_areas(pre_df, post_df)
    print_word_associations(pre_df, post_df)
    print_open_feedback(post_df)


main()

Loaded 14 pre-survey and 14 post-survey responses.

LEARNING OUTCOMES  (self-assessed Likert 1-5, Mann-Whitney U, two-sided)
Concept                                              n_pre   M_pre(SD) n_post  M_post(SD)       p
--------------------------------------------------------------------------------
L1. Machine learning                                    14  3.14(1.03)     14  3.86(1.17)   0.066
L2. Three ML paradigms and their differences**          14  1.79(1.05)     14  3.21(0.97)   0.002
L3. Where each ML paradigm can be applied**             14  1.64(0.93)     14  3.07(1.27)   0.003
L4. Data and its importance in ML                       13  2.46(1.27)     13  3.54(1.45)   0.063
L5. KNN**                                               12  1.25(0.87)     13  3.23(1.54)   0.001
L6. Linear regression**                                 13  1.23(0.60)     13  2.92(1.50)   0.003
L7. Reinforcement learning*                             13  1.85(1.14)     14  3.14(1.23)   0.014
L8. Explor